# NVIDIA ModelOpt Quantization Aware Training (QAT) Walkthrough

**Quantization Aware Training (QAT)** is a method that simulates the effects of quantization during neural network post-training to preserve accuracy when deploying models in very-low-precision formats. Unlike post-training quantization, QAT inserts "fake quantization" nodes into the computational graph, mimicking the rounding and clamping operations that occur during actual quantization. This allows the model to adapt its weights and activations to mitigate accuracy loss.

This notebook demonstrates how to apply Quantization Aware Training (QAT) to an LLM, Meta's Llama-3.1-8b in this case with NVIDIA's TensorRT Model Optimizer (ModelOpt) QAT toolkit. We walk through downloading and loading the model, calibrating it using an example dataset, specifically CNN/DailyMail dataset sample, applying NVFP4 quantization, generating outputs, and exporting the quantized model.

## Installing Prerequisites and Dependancies

If you haven't already, install the required dependencies for this notebook. Key dependancies include:

- nvidia-modelopt
- torch
- transformers
- jupyterlab

This repo contains a `examples/llm_qat/notebooks/requirements.txt` file that can be used to install all required dependancies.

In [ ]:
!pip install -r requirements.txt

## Setting HuggingFace Token and Model for Download

Set the HF_TOKEN environment variable making sure to update it to include you token (eg. `%env HF_TOKEN=hf_abdxyz...`)

In [ ]:
%env HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>

As mentioned above, we will use Meta's **Llamma-3.1-8B-Instruct** in this example

In [1]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

## Import Required Libraries

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from modelopt.torch.utils.dataset_utils import create_forward_loop, get_dataset_dataloader

## Download and Load Model and Tokenizer

In [3]:
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Load Calibration Datset

In [4]:

# Calibration dataloader
calib_loader = get_dataset_dataloader(
    dataset_name="cnn_dailymail",
    tokenizer=tokenizer,
    batch_size=8,
    num_samples=512,
    device="cuda"
)

forward_loop = create_forward_loop(dataloader=calib_loader)

/home/fghodsian/.venv/jupyter/lib/python3.12/site-packages/modelopt/torch/utils/dataset_utils.py:157: UserWarning: Tokenizer with the right padding_side may impact calibration accuracy. Recommend set to left
  warn(


## Set Quatization Config and Quantize the Model

Applying QAT with Model Optimizer is fairly straightforward. QAT supports the same quantization formats as the PTQ workflow, including key formats such as FP8, NVFP4, MXFP4, INT8, and INT4. In this example, we are using the default NVFP4 config which quantizes weight and activation to NVFP4 format. 


In [5]:
import modelopt.torch.quantization as mtq

config = mtq.NVFP4_DEFAULT_CFG

# # Define forward loop for calibration
# def forward_loop(model):
#     for data in calib_set:
#         model(data)

# quantize the model and prepare for QAT
model = mtq.quantize(model, config, forward_loop) 


Registered <class 'transformers.models.llama.modeling_llama.LlamaAttention'> to _QuantAttention for KV Cache quantization
Inserted 771 quantizers


100%|██████████████████████████████████████████████████████████████████████████████| 64/64 [01:13<00:00,  1.15s/it]


In [6]:
import modelopt.torch.opt as mto
torch.save(mto.modelopt_state(model), "modelopt_quantizer_states.pt")

In [7]:
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

# Load training dataset (for demonstration, use cnn_dailymail "train" split)
train_dataset = load_dataset("cnn_dailymail", '3.0.0', split="train[:1000]")  # Smaller subset for example

def preprocess_function(examples):
    # Concatenate the article and highlights for training
    inputs = [a + " " + h for a, h in zip(examples["article"], examples["highlights"])]
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)
    model_inputs["labels"] = model_inputs["input_ids"].copy()  # Language modeling: teacher-forced
    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)

# Data collator (for causal language modeling)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qat_model_output",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    learning_rate=1e-5,             # As recommended for QAT in README
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


/tmp/ipykernel_2585829/1505564370.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [9]:
trainer.train()

Step,Training Loss
50,0.268200
100,0.250100


TrainOutput(global_step=126, training_loss=0.25205686735728433, metrics={'train_runtime': 274.3474, 'train_samples_per_second': 7.29, 'train_steps_per_second': 0.459, 'total_flos': 4.6110257184768e+16, 'train_loss': 0.25205686735728433, 'epoch': 2.0})

In [11]:
# Save quantizer state for later resume/deploy
import modelopt.torch.opt as mto
torch.save(mto.modelopt_state(model), "modelopt_quantizer_states.pt")

# Save the final weights
trainer.save_model("./qat_model_output")